### Build a Question Answering application over a Graph Database

In [ ]:
import os
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

In [3]:
from neo4j import GraphDatabase

import dotenv
import os
from neo4j import GraphDatabase

load_status = dotenv.load_dotenv("Neo4j-1a7c6afa-Created-2026-05-01.txt")
if load_status is False:
    raise RuntimeError('Environment variables not loaded.')
URI = "neo4j+ssc://1a7c6afa.databases.neo4j.io"
AUTH = ("1a7c6afa", "FIewN75x6WJHEHShiW4AUrRaMff16f9UAthszzYKVY0")

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()
    
with driver.session(database="1a7c6afa") as session:
    result = session.run("RETURN 1 AS test")
    print(result.single())



C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\370634756.py:16: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="1a7c6afa") as session:


<Record test=1>


In [5]:
def run_query(query, params=None):
    with driver.session(database='1a7c6afa') as session:
        result = session.run(query, params or {})
        return [record.data() for record in result]

# Example
print(run_query("MATCH (n) RETURN n LIMIT 5"))

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3588860134.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database='1a7c6afa') as session:


[{'n': {}}, {'n': {}}, {'n': {}}, {'n': {'POB': 'SA', 'name': 'Elon MUSK', 'yob': 1976}}, {'n': {'name': 'Telsa'}}]


In [7]:
## Dataset Moview 
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [8]:
moview_query

"\nLOAD CSV WITH HEADERS FROM\n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE(m:Movie{id:row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director in split(row.director, '|') | \n    MERGE (p:Person {name:trim(director)})\n    MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor in split(row.actors, '|') | \n    MERGE (p:Person {name:trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre in split(row.genres, '|') | \n    MERGE (g:Genre {name:trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g))\n\n\n"

In [9]:
run_query(moview_query)

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3588860134.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database='1a7c6afa') as session:


[]

In [11]:
def get_labels():
    return run_query("CALL db.labels()")

def get_relationships():
    return run_query("CALL db.relationshipTypes()")

print(get_labels())
print(get_relationships())

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3588860134.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database='1a7c6afa') as session:


[{'label': 'CEO'}, {'label': 'employee'}, {'label': 'Company'}, {'label': 'Person'}, {'label': 'Moview'}, {'label': 'User'}, {'label': 'Post'}, {'label': 'Movie'}, {'label': 'Genre'}]
[{'relationshipType': 'ACTED_IN'}, {'relationshipType': 'POSTED'}, {'relationshipType': 'FRIEND'}, {'relationshipType': 'LIKES'}, {'relationshipType': 'DIRECTED'}, {'relationshipType': 'IN_GENRE'}]


In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [19]:
from langchain_groq import ChatGroq
llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023DD9011950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023DD9012350>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [28]:
def generate_cypher(question):
    prompt = f"""
You are a Neo4j expert.

Convert the following question into a Cypher query.

STRICT RULES:
- Output ONLY the Cypher query
- Do NOT include explanation
- Do NOT include text before or after
- Query must start with MATCH / RETURN

Question:
{question}
"""
    return llm.invoke(prompt).content.strip()

def clean_cypher(query):
    query = query.strip()

    if not query.upper().startswith(("MATCH", "RETURN", "WITH", "CALL")):
        raise ValueError(f"❌ Invalid Cypher generated:\n{query}")

    return query

In [21]:
def run_query(query):
    with driver.session(database="1a7c6afa") as session:
        result = session.run(query)
        return [r.data() for r in result]

In [29]:
def answer_question(question):
    cypher = generate_cypher(question)
    cypher = clean_cypher(cypher)

    data = run_query(cypher)

    final_prompt = f"""
Answer based on this data:
{data}

Question: {question}
"""
    return llm.invoke(final_prompt).content

In [30]:

print(answer_question("Who was the director of the movie Casino"))

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3164884891.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="1a7c6afa") as session:


I couldn't find any information on the director of the movie "Casino" in the provided data. However, I can provide the answer based on general knowledge.

The director of the movie "Casino" (1995) is Martin Scorsese.


In [31]:

print(answer_question("Who were the actors of the movie Casino?"))

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3164884891.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="1a7c6afa") as session:


The main actors of the movie "Casino" (1995) include:

- Robert De Niro as Ace Rothstein
- Sharon Stone as Ginger McKenna
- Joe Pesci as Tommy DeVito

Other notable actors in the film include:

- James Woods as Lester Diamond
- Don Rickles as Billy Sherbert
- Frank Vincent as Frank 'Left Hand' Marino
- Pasquale Cajano as Remo Gaggi
- John Fiedler as Jerry
- Al Roker as Himself


In [32]:
print(answer_question("How many artists are there?"))

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3164884891.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="1a7c6afa") as session:


There are 0 artists.


In [33]:
print(answer_question("How many movies has Tom Hanks acted in?"))

C:\Users\bhara\AppData\Local\Temp\ipykernel_36576\3164884891.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session(database="1a7c6afa") as session:


Since the given data only contains one row with a count of 0, and there's no specific information about Tom Hanks, it appears that the data is incomplete or does not contain any information about Tom Hanks' movie count. Therefore, I cannot provide an accurate answer.
